# 07. Enhance Dataset
Merge the new `sentiment_statements-data.csv` with existing `sentfin_strict.csv` to create an enhanced, deduplicated dataset.

In [1]:
import pandas as pd


In [2]:
# Load new dataset (no header, latin1 encoding, sentiment first)
new_df = pd.read_csv('../data/raw/sentiment_statements-data.csv', header=None, encoding='latin1', names=['sentiment', 'text'])

# Load existing strictly processed dataset (header present, text first)
old_df = pd.read_csv('../data/processed/sentfin_strict.csv')

print(f"New dataset shape: {new_df.shape}")
print(f"Old dataset shape: {old_df.shape}")


New dataset shape: (4846, 2)
Old dataset shape: (9518, 2)


In [3]:
# Standardize sentiment labels (lowercase, strip whitespace)
new_df['sentiment'] = new_df['sentiment'].str.lower().str.strip()

# Drop rows where 'text' or 'sentiment' is missing or not a string
new_df = new_df.dropna(subset=['text', 'sentiment'])
old_df = old_df.dropna(subset=['text', 'sentiment'])

# Ensure target columns order matches (text, sentiment)
new_df = new_df[['text', 'sentiment']]

print("New dataset sentment distribution:")
print(new_df['sentiment'].value_counts())


New dataset sentment distribution:
sentiment
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64


In [4]:
# Deduplicate within new dataset
new_df_unique = new_df.drop_duplicates(subset=['text'])
print(f"New dataset unique rows: {new_df_unique.shape[0]} (Dropped {new_df.shape[0] - new_df_unique.shape[0]} duplicates)")

# Concatenate old and new datasets
merged_df = pd.concat([old_df, new_df_unique], ignore_index=True)
print(f"Combined dataset shape: {merged_df.shape}")

# Deduplicate globally based on text to prevent any overlap
merged_df_unique = merged_df.drop_duplicates(subset=['text'])
print(f"Final deduplicated dataset shape: {merged_df_unique.shape[0]} (Dropped {merged_df.shape[0] - merged_df_unique.shape[0]} global duplicates)")

print("\nFinal sentiment distribution:")
print(merged_df_unique['sentiment'].value_counts())


New dataset unique rows: 4838 (Dropped 8 duplicates)
Combined dataset shape: (14356, 2)
Final deduplicated dataset shape: 14297 (Dropped 59 global duplicates)

Final sentiment distribution:
sentiment
neutral     6279
positive    4714
negative    3304
Name: count, dtype: int64


In [5]:
output_path = '../data/processed/merged_sentiment_data.csv'
merged_df_unique.to_csv(output_path, index=False)
print(f"Enhanced dataset successfully exported to {output_path}")


Enhanced dataset successfully exported to ../data/processed/merged_sentiment_data.csv
